# Notebook 3 – Décodage de la grille graphique
**Projet Computer Vision IG.2405 – 2026**

Ce notebook traite la lecture des éléments **graphiques** de la page 1 :
- **Student ID** : grille 5 colonnes × 10 lignes (une case cochée par colonne = un chiffre)
- **Groupe** : grille 3 colonnes × 10 lignes (2 chiffres + 1 lettre)
- **Conditions d'examen** : 5 cases YES/NO
- **Cryptogramme** : empreinte graphique du formulaire

> Méthodes bas niveau uniquement : morphologie mathématique, composantes connexes, analyse de densité d'encre.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from utils.grid_decoder import (
    normalize_page, get_roi,
    ROI_STUDENT_ID, ROI_GROUP_GRID, ROI_SIGNATURE, ROI_CRYPTO,
    STUDENT_ID_ROWS, STUDENT_ID_COLS,
    read_student_id, read_group, read_conditions, extract_cryptogram,
)
from utils.checkbox_reader import (
    preprocess_for_checkbox, ink_ratio, has_x_pattern,
    split_grid, read_grid_one_per_col, is_checkbox_checked,
)
from utils.form_aligner import deskew

FORM_DIR = os.path.join('PROJECT 2026 -DATABASE-20260518', 'FORM1')

# Image de calibration (Student ID = 62445, Group = G02B)
img_files = sorted([f for f in os.listdir(FORM_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
SAMPLE = os.path.join(FORM_DIR, img_files[0])
print('Image :', img_files[0])

img_raw  = cv2.imread(SAMPLE)
img_ds   = deskew(img_raw)
img_norm = normalize_page(img_ds)
print(f'Image normalisée : {img_norm.shape[1]}×{img_norm.shape[0]} px')

## 1. Extraction de la grille Student ID

La grille Student ID occupe la zone `ROI_STUDENT_ID = (725, 247, 155, 330)` dans l'image normalisée.

Structure : **5 colonnes** (un chiffre par colonne), **10 lignes** (digits 0–9).
L'étudiant noircit une case par colonne.

In [ ]:
roi_sid = get_roi(img_norm, ROI_STUDENT_ID)

# Découper en cellules
cells = split_grid(roi_sid, rows=STUDENT_ID_ROWS, cols=STUDENT_ID_COLS)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(roi_sid, cv2.COLOR_BGR2RGB))
axes[0].set_title('ROI grille Student ID')
axes[0].axis('off')

# Visualiser la grille segmentée
grid_vis = roi_sid.copy()
h, w = roi_sid.shape[:2]
cell_h, cell_w = h // STUDENT_ID_ROWS, w // STUDENT_ID_COLS

for r in range(STUDENT_ID_ROWS + 1):
    cv2.line(grid_vis, (0, r * cell_h), (w, r * cell_h), (0, 255, 0), 1)
for c in range(STUDENT_ID_COLS + 1):
    cv2.line(grid_vis, (c * cell_w, 0), (c * cell_w, h), (0, 0, 255), 1)

axes[1].imshow(cv2.cvtColor(grid_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Grille segmentée ({STUDENT_ID_COLS}×{STUDENT_ID_ROWS})')
axes[1].axis('off')

plt.suptitle('Grille Student ID', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Mesure de la densité d'encre par cellule

Pour chaque cellule $(r, c)$, on calcule le ratio d'encre :

$$\text{ink\_ratio}(r,c) = \frac{\#\text{pixels encre}}{\#\text{pixels total}}$$

La case cochée est celle avec le **ratio maximal** par colonne.

In [ ]:
# Calculer les ratios d'encre pour toutes les cellules
ratios = np.zeros((STUDENT_ID_ROWS, STUDENT_ID_COLS))

for r in range(STUDENT_ID_ROWS):
    for c in range(STUDENT_ID_COLS):
        binary = preprocess_for_checkbox(cells[r][c])
        margin = max(2, int(min(binary.shape) * 0.10))
        inner  = binary[margin:-margin, margin:-margin] if (
            binary.shape[0] > 2*margin and binary.shape[1] > 2*margin) else binary
        ratios[r, c] = ink_ratio(inner)

# Afficher la heatmap des ratios
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(ratios, cmap='hot', aspect='auto')
axes[0].set_xlabel('Colonne (chiffre du Student ID)')
axes[0].set_ylabel('Ligne (valeur 0-9)')
axes[0].set_xticks(range(STUDENT_ID_COLS))
axes[0].set_yticks(range(STUDENT_ID_ROWS))
axes[0].set_yticklabels(range(STUDENT_ID_ROWS))
axes[0].set_title('Densité d\'encre par cellule')
plt.colorbar(im, ax=axes[0])

# Afficher le chiffre détecté par colonne
detected = read_grid_one_per_col(roi_sid, rows=STUDENT_ID_ROWS, cols=STUDENT_ID_COLS)
print('Chiffres détectés par colonne :', detected)
student_id = int(''.join(str(d) for d in detected)) if None not in detected else None
print(f'Student ID reconstruit : {student_id}')

# Marquer les cases détectées
marked = ratios.copy()
for c, r in enumerate(detected):
    if r is not None:
        marked[r, c] += 0.5

axes[1].imshow(marked, cmap='coolwarm', aspect='auto')
axes[1].set_xlabel('Colonne (chiffre du Student ID)')
axes[1].set_ylabel('Ligne (valeur 0-9)')
axes[1].set_xticks(range(STUDENT_ID_COLS))
axes[1].set_yticks(range(STUDENT_ID_ROWS))
axes[1].set_yticklabels(range(STUDENT_ID_ROWS))
axes[1].set_title(f'Cases détectées → Student ID = {student_id}')

plt.suptitle('Lecture de la grille Student ID', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Lecture du Student ID via la fonction principale

In [ ]:
sid = read_student_id(img_norm)
print(f'Student ID lu : {sid}')

# Vérifier la cohérence avec le nom de fichier
expected = img_files[0].split('_')[-1].split('.')[0]
print(f'Student ID attendu (nom fichier) : {expected}')
print(f'Correspondance : {str(sid) == expected}')

## 4. Lecture du Groupe

In [ ]:
roi_grp = get_roi(img_norm, ROI_GROUP_GRID)

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
axes[0].imshow(cv2.cvtColor(roi_grp, cv2.COLOR_BGR2RGB))
axes[0].set_title('ROI Grille Groupe')
axes[0].axis('off')

# Résultat
group = read_group(img_norm)
print(f'Groupe lu : {group}')

axes[1].imshow(cv2.cvtColor(roi_grp, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Groupe détecté : {group}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Lecture des conditions d'examen (cases YES/NO)

5 conditions : Notes de cours / Notes manuscrites / Ordinateur / Calculatrice / Brouillon

Méthode : détection d'un **carré plein** (■ = YES) vs case vide (NO) via densité d'encre.

In [ ]:
from utils.grid_decoder import CONDITIONS, COND_Y_YESNO, COND_CHECKBOX_W
from utils.checkbox_reader import is_filled_square

conditions = read_conditions(img_norm)
print('Conditions d\'examen :')
labels = ['Notes de cours', 'Notes manuscrites', 'Ordinateur', 'Calculatrice', 'Brouillon']
for label, (key, val) in zip(labels, conditions.items()):
    status = 'YES' if val else 'NO'
    detail = f'(max={val})' if val > 1 else ''
    print(f'  {label:20s} : {status} {detail}')

# Visualiser les cases YES/NO
y0, y1 = COND_Y_YESNO
vis_cond = img_norm.copy()

for i, (x_yes, x_no, *_) in enumerate(CONDITIONS):
    w = COND_CHECKBOX_W
    # Case YES
    color_yes = (0, 255, 0) if list(conditions.values())[i] else (200, 200, 200)
    cv2.rectangle(vis_cond, (x_yes, y0), (x_yes + w, y1), color_yes, 2)
    # Case NO
    color_no = (0, 0, 255) if not list(conditions.values())[i] else (200, 200, 200)
    cv2.rectangle(vis_cond, (x_no, y0), (x_no + w, y1), color_no, 2)

# Rogner sur la zone conditions
margin = 20
crop = vis_cond[max(0, y0 - margin):y1 + margin, :]

plt.figure(figsize=(12, 3))
plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
plt.title('Cases YES (vert) / NO (rouge) détectées')
plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Détection des cases MCQ dans les pages d'examen

Dans les pages d'examen, les réponses MCQ sont des cases à cocher marquées d'un **X**.

Détection : densité d'encre + analyse des **projections diagonales**.

In [ ]:
import fitz
from utils.pdf_utils import pdf_to_images
from utils.exam_parser import detect_question_blocks, _find_mcq_checkboxes, _is_checked_box

pdf_files = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.pdf')])
if pdf_files:
    sample_pdf = os.path.join(FORM_DIR, pdf_files[0])
    pages = pdf_to_images(sample_pdf, dpi=150)
    print(f'PDF : {pdf_files[0]} → {len(pages)} pages')

    # Travailler sur la page 5 (première page d'examen, index 4)
    EXAM_START = 4
    if len(pages) > EXAM_START:
        exam_page = normalize_page(pages[EXAM_START])

        # Détecter les blocs de questions
        blocks = detect_question_blocks(exam_page)
        print(f'Blocs de questions détectés : {len(blocks)}')

        # Visualiser le premier bloc avec ses cases MCQ
        if blocks:
            y_start, y_end = blocks[0]
            block_img = exam_page[y_start:y_end, :]

            # Trouver les cases MCQ
            header_cut = max(20, int(block_img.shape[0] * 0.22))
            content = block_img[header_cut:, :]
            boxes = _find_mcq_checkboxes(content)
            print(f'Cases MCQ dans le bloc 1 : {len(boxes)}')

            vis_block = content.copy()
            for x, y, w, h in boxes:
                checked = _is_checked_box(content, x, y, w, h)
                color = (0, 255, 0) if checked else (0, 0, 255)
                cv2.rectangle(vis_block, (x, y), (x + w, y + h), color, 2)

            plt.figure(figsize=(10, 4))
            plt.imshow(cv2.cvtColor(vis_block, cv2.COLOR_BGR2RGB))
            plt.title('Cases MCQ : cochées (vert) / non cochées (rouge)')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

## 7. Extraction du cryptogramme

Le cryptogramme est une empreinte graphique identique sur toutes les pages d'un même formulaire.
On vérifie sa cohérence par **corrélation croisée normalisée (NCC)** entre les pages.

In [ ]:
crypto = extract_cryptogram(img_norm)
print(f'Dimensions cryptogramme : {crypto.shape[1]}×{crypto.shape[0]} px')

plt.figure(figsize=(3, 2))
plt.imshow(cv2.cvtColor(crypto, cv2.COLOR_BGR2RGB))
plt.title('Cryptogramme extrait')
plt.axis('off')
plt.tight_layout()
plt.show()

if pdf_files and len(pages) > 1:
    from utils.page1_parser import compare_cryptograms
    crypto_pages = []
    for pg in pages[1:]:
        norm_pg = normalize_page(pg)
        c = extract_cryptogram(norm_pg)
        if c is not None and c.size > 0:
            crypto_pages.append(c)

    valid = compare_cryptograms(crypto_pages, crypto)
    print(f'Cryptogrammes cohérents sur {len(crypto_pages)} pages : {valid}')

    # Afficher quelques cryptogrammes de pages différentes
    n_show = min(4, len(crypto_pages))
    if n_show > 0:
        fig, axes = plt.subplots(1, n_show + 1, figsize=(3 * (n_show + 1), 2))
        axes[0].imshow(cv2.cvtColor(crypto, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Page 1 (réf.)')
        axes[0].axis('off')
        for i in range(n_show):
            axes[i + 1].imshow(cv2.cvtColor(crypto_pages[i], cv2.COLOR_BGR2RGB))
            axes[i + 1].set_title(f'Page {i+2}')
            axes[i + 1].axis('off')
        plt.suptitle('Cryptogrammes de toutes les pages', fontsize=11)
        plt.tight_layout()
        plt.show()

## Bilan

| Élément | Méthode | Résultat |
|---|---|---|
| Student ID | Ink ratio par cellule, max par colonne | Entier à 5 chiffres |
| Groupe | Idem (3 colonnes) | Chaîne 'GxxY' |
| Conditions YES/NO | Carré plein vs case vide | Dict 0/1/N |
| Cases MCQ | Composantes connexes + ratio X | Dict lettre → 1/None |
| Cryptogramme | NCC inter-pages | Booléen cohérence |

**Prochaine étape** → `04_signature_authentication.ipynb` : vérification des signatures.